In [2]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt
import os

# ==========================================
# 1. دوال استخراج الزوايا والتماثل (نفسها من المهمة 3)
# ==========================================
def calculate_joint_angle(A, B, C):
    if A is None or B is None or C is None: return None
    u, v = np.array(A) - np.array(B), np.array(C) - np.array(B)
    norm_u, norm_v = np.linalg.norm(u), np.linalg.norm(v)
    if norm_u < 1e-6 or norm_v < 1e-6: return None
    cos_theta = np.clip(np.dot(u, v) / (norm_u * norm_v), -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_theta)))

def get_landmark_coords(landmarks, landmark_id, visibility_threshold=0.5):
    if landmarks is None: return None
    lm = landmarks.landmark[landmark_id]
    return (lm.x, lm.y) if lm.visibility >= visibility_threshold else None

def calculate_symmetry_index(left_val, right_val):
    if left_val is None or right_val is None: return None
    denominator = left_val + right_val
    if denominator < 1e-6: return None
    return float((2.0 * abs(left_val - right_val)) / denominator * 100.0)

def extract_frame_features(landmarks):
    features = {}
    if landmarks is None:
        return {k: None for k in ["L_Hip", "L_Knee", "L_Ankle", "R_Hip", "R_Knee", "R_Ankle", 
                                  "Knee_Symmetry", "Hip_Symmetry", "Ankle_Symmetry"]}
    
    L_hip, R_hip = get_landmark_coords(landmarks, 23), get_landmark_coords(landmarks, 24)
    L_knee, R_knee = get_landmark_coords(landmarks, 25), get_landmark_coords(landmarks, 26)
    L_ankle, R_ankle = get_landmark_coords(landmarks, 27), get_landmark_coords(landmarks, 28)
    L_shoulder, R_shoulder = get_landmark_coords(landmarks, 11), get_landmark_coords(landmarks, 12)
    L_heel, R_heel = get_landmark_coords(landmarks, 29), get_landmark_coords(landmarks, 30)
    
    features["L_Knee"] = calculate_joint_angle(L_hip, L_knee, L_ankle)
    features["R_Knee"] = calculate_joint_angle(R_hip, R_knee, R_ankle)
    features["L_Hip"] = calculate_joint_angle(L_shoulder, L_hip, L_knee)
    features["R_Hip"] = calculate_joint_angle(R_shoulder, R_hip, R_knee)
    features["L_Ankle"] = calculate_joint_angle(L_knee, L_ankle, L_heel)
    features["R_Ankle"] = calculate_joint_angle(R_knee, R_ankle, R_heel)
    
    features["Knee_Symmetry"] = calculate_symmetry_index(features["L_Knee"], features["R_Knee"])
    features["Hip_Symmetry"] = calculate_symmetry_index(features["L_Hip"], features["R_Hip"])
    features["Ankle_Symmetry"] = calculate_symmetry_index(features["L_Ankle"], features["R_Ankle"])
    return features

# ==========================================
# 2. دالة مرشح Butterworth Low-Pass (تنفيذ الشرط FR3)
# ==========================================
def apply_butterworth_filter(data_series, fps, cutoff=6.0, order=4):
    """
    يطبق مرشح Butterworth لإزالة الاهتزاز عالي التردد (Jitter) من الإشارة.
    التردد 6.0 Hz هو المعيار الذهبي لتحليل مشية الإنسان (حسب أدبيات Biomechanics).
    """
    nyquist = 0.5 * fps
    normal_cutoff = cutoff / nyquist
    
    # تصميم المرشح
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    
    # تطبيق المرشح بشكل أمامي وخلفي (filtfilt) للحفاظ على طور الإشارة (Zero-phase delay)
    # ملاحظة: filtfilt لا يقبل قيم NaN، لذا يجب استدعاء هذه الدالة بعد معالجة القيم المفقودة
    filtered_data = filtfilt(b, a, data_series)
    return filtered_data

# ==========================================
# 3. خط الأنابيب الرئيسي مع التنقية والحفظ
# ==========================================
def process_and_save_gait_data(video_path, output_csv):
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=False, model_complexity=1, min_detection_confidence=0.5, min_tracking_confidence=0.5)
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    raw_features = []
    print(f"🎬 1/3 جاري استخراج الملامح الخام من {total_frames} إطار...")
    
    frame_idx = 0
    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
        
        results = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        frame_feats = extract_frame_features(results.pose_landmarks)
        frame_feats["Frame"] = frame_idx
        frame_feats["Timestamp_sec"] = frame_idx / fps
        raw_features.append(frame_feats)
        frame_idx += 1
        
    cap.release()
    pose.close()
    
    df_raw = pd.DataFrame(raw_features)
    
    print("🧹 2/3 جاري معالجة البيانات المفقودة وتطبيق مرشح Butterworth (FR3)...")
    
    # أ) معالجة القيم المفقودة (Interpolation) لأن مرشح filtfilt لا يعمل مع NaN
    # نستخدم الاستكمال الخطي للأمام والخلف لملء الفجوات الصغيرة الناتجة عن اختفاء المفصل
    cols_to_filter = ["L_Hip", "L_Knee", "L_Ankle", "R_Hip", "R_Knee", "R_Ankle", 
                      "Knee_Symmetry", "Hip_Symmetry", "Ankle_Symmetry"]
    
    df_clean = df_raw.copy()
    for col in cols_to_filter:
        # استكمال القيم المفقودة خطياً
        df_clean[col] = df_clean[col].interpolate(method='linear', limit_direction='both')
        # تطبيق مرشح Butterworth
        df_clean[f"{col}_Filtered"] = apply_butterworth_filter(df_clean[col].values, fps, cutoff=6.0)
    
    # ب) إعادة حساب التماثل بناءً على القيم المفلترة (أكثر دقة)
    df_clean["Knee_Symmetry_Filtered"] = df_clean.apply(
        lambda row: calculate_symmetry_index(row["L_Knee_Filtered"], row["R_Knee_Filtered"]), axis=1
    )
    
    # ج) اختيار الأعمدة النهائية النظيفة فقط للحفظ
    final_cols = ["Frame", "Timestamp_sec", 
                  "L_Knee_Filtered", "R_Knee_Filtered", "Knee_Symmetry_Filtered",
                  "L_Hip_Filtered", "R_Hip_Filtered", "Hip_Symmetry_Filtered",
                  "L_Ankle_Filtered", "R_Ankle_Filtered", "Ankle_Symmetry_Filtered"]
    
    df_final = df_clean[final_cols]
    
    print(f"💾 3/3 جاري حفظ البيانات النظيفة في: {output_csv}")
    df_final.to_csv(output_csv, index=False)
    
    print("\n✅ تم الانتهاء بنجاح!")
    print("📊 ملخص البيانات النهائية الجاهزة للتدريب (XGBoost):")
    print(df_final.describe().round(2))
    
    # تحليل التماثل النهائي
    print("\n⚖️ تحليل التماثل النهائي (بعد التنقية):")
    for col in ["Knee_Symmetry_Filtered", "Hip_Symmetry_Filtered", "Ankle_Symmetry_Filtered"]:
        mean_si = df_final[col].mean()
        status = "⚠️ غير متماثل (>10%)" if mean_si > 10 else "✅ متماثل"
        print(f"  {col.replace('_Filtered', '')}: متوسط = {mean_si:.2f}%  {status}")

# ==========================================
# 4. التشغيل
# ==========================================
if __name__ == "__main__":
    VIDEO_PATH = r"C:\Users\hacker\Desktop\machine_learnig\Gaitly_AI\viduo\WhatsApp Video 2026-10-17 at 1.13.06 AM.mp4"
    OUTPUT_CSV = "gait_features_clean.csv" # هذا هو الملف الذي سنستخدمه في الأسابيع 5-6
    
    if os.path.exists(VIDEO_PATH):
        process_and_save_gait_data(VIDEO_PATH, OUTPUT_CSV)
    else:
        print(f"❌ يرجى التأكد من وجود ملف الفيديو: {VIDEO_PATH}")

🎬 1/3 جاري استخراج الملامح الخام من 433 إطار...
🧹 2/3 جاري معالجة البيانات المفقودة وتطبيق مرشح Butterworth (FR3)...
💾 3/3 جاري حفظ البيانات النظيفة في: gait_features_clean.csv

✅ تم الانتهاء بنجاح!
📊 ملخص البيانات النهائية الجاهزة للتدريب (XGBoost):
        Frame  Timestamp_sec  L_Knee_Filtered  R_Knee_Filtered  \
count  433.00         433.00           433.00           433.00   
mean   216.00           7.20           134.36           123.99   
std    125.14           4.17            42.38            33.66   
min      0.00           0.00            45.96            57.85   
25%    108.00           3.60            73.40            85.06   
50%    216.00           7.20           151.94           131.08   
75%    324.00          10.80           171.24           152.84   
max    432.00          14.40           185.70           181.54   

       Knee_Symmetry_Filtered  L_Hip_Filtered  R_Hip_Filtered  \
count                  433.00          433.00          433.00   
mean                    